# 02 — PDF Structure Exploration

Inspect the built-in Table of Contents, text availability, page statistics, and heading candidates.
The built-in TOC remains the authoritative hierarchy for downstream chunking.

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "intelligent_scada").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Intelligent SCADA project root.")

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from intelligent_scada.config import Settings

settings = Settings.from_env(project_root=project_root, env_override=True).for_default_document()
settings.ensure_output_dirs()


In [ ]:
from collections import Counter

from intelligent_scada.ingestion.pdf_analyzer import analyze_pdf

structure = analyze_pdf(settings.require_default_pdf(), settings.pdf_structure_path)
level_counts = Counter(entry["level"] for entry in structure["toc"])

print(f"Pages: {structure['total_pages']}")
print(f"TOC entries: {structure['toc_entries']}")
print(f"TOC levels: {dict(sorted(level_counts.items()))}")
print(f"Empty pages: {len(structure['empty_pages'])}")
print(f"Heading candidates: {structure['heading_candidates_count']}")
print(f"Output: {settings.pdf_structure_path}")


In [ ]:
# Exploratory preview: first 20 TOC entries
for entry in structure["toc"][:20]:
    indent = "  " * (entry["level"] - 1)
    print(f"{indent}- {entry['title']} (page {entry['page']})")
